# 03. XLM-RoBERTa large

The same tokenizer and pre-training data (CC100, 100 languages) as XLM-R base, but 24 layers,
hidden size 1024 and 560M parameters instead of 12 / 768 / 280M. Larger models usually gain most
on rare words, which is where our errors are (place names not seen in training).

The data and training settings come from `ner/recipes.py` (decided in notebooks 01 and 02); here we
only change what is specific to this model:

- **lr 1e-5 or 2e-5** instead of 3e-5. Large models are known to collapse (predict only "O")
  with a too high learning rate early in training; watch the training loss for a sudden jump.
- **microbatch 2 with accumulation** preserving the recipe's effective batch, gradient checkpointing,
  and evaluation/inference batch 4. The preflight checks actual GPU memory; T4 fit is not assumed.

Steps:

1. **Learning rate on fold 0.** Two values (stage 1; stage 2 scales with it, as in the recipe); pick
   the one with the higher `half_B` F1.
2. **Folds 1 and 2** with that learning rate. The mean `cv` score over the three folds is the
   cross-validation score of this model, on the same folds as every other model and the Assignment 3
   RNNs.
3. **Fold ensemble.** The three fold models' probabilities on the test posts are averaged. That gives
   development scores on half B, report-only gold scores, and a Kaggle submission.

On a T4 one fold takes about 90-120 minutes (the `minutes` column shows the measured time).

### Colab setup and preservation

Select a GPU runtime, then run top to bottom. This notebook checks out the verified training revision in `REPO_REF`,
installs the shared environment, mounts Drive and authenticates W&B. Set `WANDB_ENTITY` for team logging.
The branch must be published before opening it in Colab; `REPO_REF` can also be a frozen commit SHA.
Use a fresh runtime when changing code or dependencies.

Every completed run preserves weights, tokenizer, validation/test probabilities, row identities,
metrics, curves and submission on Drive. Interrupted runs resume from the last saved checkpoint;
finished runs are reused only after artifact verification. `NER_SMOKE=1` uses a separate Drive folder
and disables W&B. The GPU preflight measures full-window memory before launching the folds.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"
REPO_REF = "357bef20057ade30b343478d214172fa22b06651"  # verified training code; change for a new experiment
WANDB_PROJECT = "dl-a4-ner"  # None disables W&B
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "models-kyiv-school-of-economics")
DRIVE_ROOT = "/content/drive/MyDrive/dl-a4-ner"

if "google.colab" in sys.modules:
    if "ner.train" in sys.modules:
        raise RuntimeError(
            "Restart the runtime before changing the training checkout or dependencies"
        )
    repo = Path("/content/deep-learning-assignment4")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    if subprocess.check_output(
        ["git", "-C", str(repo), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("The Colab checkout has edits; preserve them before selecting REPO_REF")
    subprocess.run(["git", "-C", str(repo), "fetch", "-q", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    os.chdir(repo)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "transformers/requirements-colab.txt"],
        check=True,
    )
    artifact_root = Path(DRIVE_ROOT)
else:
    if Path.cwd().name == "transformers":
        os.chdir("..")
    artifact_root = Path("transformers")
sys.path.insert(0, os.getcwd())
from ner.experiment import setup

OUT, WANDB_PROJECT = setup(artifact_root / "runs", WANDB_PROJECT, WANDB_ENTITY)
print("Repository:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import dataclasses

from ner import plots, recipes
from ner.cv import cv_table, export_fold_ensemble
from ner.experiment import (
    choose_learning_rate,
    ensemble_directory,
    experiment_prefix,
    freeze_model_revision,
    record_kaggle_result,
)
from ner.preflight import memory_check
from ner.train import RunConfig, load_run, run_or_load

# Preserve the recipe's effective batch while reducing activation memory.
EFFECTIVE_BATCH = recipes.OPTIMIZATION["batch_size"] * recipes.OPTIMIZATION.get(
    "grad_accumulation", 1
)
MICROBATCH = 2
if EFFECTIVE_BATCH % MICROBATCH:
    raise ValueError("Choose a microbatch that divides the recipe's effective batch")
OVERRIDES = dict(
    batch_size=MICROBATCH,
    grad_accumulation=EFFECTIVE_BATCH // MICROBATCH,
    eval_batch_size=4,
    inference_batch_size=4,
    gradient_checkpointing=True,
)

MODEL = "FacebookAI/xlm-roberta-large"
RECIPE = RunConfig(
    name="recipe",
    data=recipes.DATA,
    **(recipes.STAGE2 or {}),
    **{**recipes.OPTIMIZATION, **OVERRIDES},
    model=MODEL,
    revision=freeze_model_revision(MODEL),
    tags=("03-xlmr-large",),
)
PREFIX = experiment_prefix("03-xlmr-large", RECIPE)
ENSEMBLE_DIR = ensemble_directory(OUT, PREFIX)
print(RECIPE)
print("Experiment:", PREFIX)


def run_name(lr, fold):
    return f"{PREFIX}-lr{lr:g}-fold{fold}"


def train(lr, fold):
    stage2_lr = lr * RECIPE.stage2_lr / RECIPE.learning_rate if RECIPE.stage2_lr else None
    cfg = dataclasses.replace(
        RECIPE, name=run_name(lr, fold), learning_rate=lr, stage2_lr=stage2_lr, fold=fold
    )
    return run_or_load(cfg, OUT, WANDB_PROJECT)


# This model is discarded; experiment runs start from the pinned pretrained weights.
memory_check(RECIPE, OUT / f"{PREFIX}-memory.json")


## 1. Learning rate on fold 0

In [ ]:
LEARNING_RATES = [1e-5, 2e-5]
for lr in LEARNING_RATES:
    train(lr, fold=0)


In [ ]:
lr_runs = {f"lr {lr:g}": load_run(OUT, run_name(lr, 0)) for lr in LEARNING_RATES}
lr_table = plots.results_table(lr_runs).round(6)
lr_table.to_csv(OUT / f"{PREFIX}-lr-comparison.csv")
lr_table


In [ ]:
plots.plot_curves(
    lr_runs,
    title="XLM-R large: learning rate on fold 0",
    save_to=None
    if (ENSEMBLE_DIR / "complete.json").exists()
    else ENSEMBLE_DIR / "figures/lr-curves.png",
)


## 2. Folds 1 and 2

Select by `half_B` development F1 among verified fold-0 runs. Ties use the lower learning rate.
The decision and all trial scores are saved beside the ensemble; gold26 remains report-only.


In [ ]:
BEST_LR, SELECTION = choose_learning_rate(OUT, run_name, LEARNING_RATES)
print("Selected:", BEST_LR, SELECTION)
for fold in (1, 2):
    train(BEST_LR, fold=fold)


In [ ]:
FOLD_RUNS = [run_name(BEST_LR, k) for k in range(3)]
fold_table = cv_table(OUT, FOLD_RUNS)
fold_table.to_csv(OUT / f"{PREFIX}-cv.csv")
fold_table.round(4)


## 3. Fold ensemble and submission

In [ ]:
best = load_run(OUT, FOLD_RUNS[0])
plots.plot_curves(
    {"fold 0": best},
    title="XLM-R large, fold 0",
    save_to=None
    if (ENSEMBLE_DIR / "complete.json").exists()
    else ENSEMBLE_DIR / "figures/fold-0-curves.png",
)
plots.plot_grad_flow(
    best,
    title="XLM-R large, fold 0",
    save_to=None
    if (ENSEMBLE_DIR / "complete.json").exists()
    else ENSEMBLE_DIR / "figures/fold-0-gradients.png",
)
ensemble_metrics, sub = export_fold_ensemble(
    OUT,
    FOLD_RUNS,
    ENSEMBLE_DIR,
    SELECTION,
    lr_table,
    wandb_project=WANDB_PROJECT,
)
print("Drive ensemble bundle:", ENSEMBLE_DIR)
ensemble_metrics


In [ ]:
# Check inference from the saved weights; the loader does not fetch the base model.
from ner.data import load_gold_sample
from ner.inference import predict_saved

restore_texts = list(load_gold_sample().text[:2])
restored_predictions, _ = predict_saved(OUT / FOLD_RUNS[0], restore_texts)
restored_predictions


### Preserve the actual Kaggle score
Submit `ENSEMBLE_DIR / "submission.csv"` to Kaggle yourself. Then fill the score and submission ID
below. This records a dated result beside the exact submission hash on Drive. `half_B` above is a
pseudo-label development score, not a Kaggle score.


In [ ]:
KAGGLE_PUBLIC_SCORE = None
KAGGLE_SUBMISSION_ID = None
if KAGGLE_PUBLIC_SCORE is not None:
    record_kaggle_result(ENSEMBLE_DIR, KAGGLE_PUBLIC_SCORE, KAGGLE_SUBMISSION_ID)


## Conclusions

- CV (mean of `cv` over folds) and its spread, `half_B`, `gold26` and the public leaderboard
  (add the submission's score to `transformers/leaderboard.csv`).
- Compared with XLM-R base (notebook 02 baseline): is this model worth its cost?
- Is the model adequate? Check that the training loss went down smoothly, that held-out loss and F1
  didn't get worse in the last epoch (overfitting), that the gradient norms of the lower layers are not
  zero (they learn) and not exploding, and that CV and `half_B` agree with the leaderboard in order.